# Tree of Thoughts (ToT)

> Notebook generated from [`examples/patterns/01_tree_of_thoughts.py`](../../examples/patterns/01_tree_of_thoughts.py).

| Field | Value |
|------|-------|
| **Dataset** | GSM8K (embedded — 5 math problems) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Per problem: winning thought with score, depth of the best path and step-by-step reasoning path. Final comparison of `beam` vs `bfs` vs `dfs` strategies.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Tree of Thoughts (ToT) — Solving GSM8K math problems
====================================================================
Pattern: SPEC-PAT-001 / prismal.agents.patterns.tree_of_thoughts

Dataset: GSM8K (Grade School Math 8K)
  • 8,500 elementary/middle-school math problems with step-by-step solutions.
  • Reference: https://huggingface.co/datasets/openai/gsm8k
  • Why: ToT shines on multi-step reasoning where you must explore and prune
    solution branches; GSM8K demands exactly that.

Pattern description:
  ToT builds a tree of thoughts (solution steps). At each node,
  generate_fn proposes N candidates; evaluate_fn scores them. The search
  strategy (beam / BFS / DFS) decides which branches to keep. It stops as soon
  as any thought reaches the quality threshold or the depth is exhausted.

Usage:
    uv run python examples/patterns/01_tree_of_thoughts.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import re

from langchain_core.messages import HumanMessage, SystemMessage

from prismal.agents.patterns.tree_of_thoughts import ToTResult, tree_of_thoughts
from prismal.core.config import get_settings
from prismal.providers.registry import ProviderRegistry

## Dataset: fixed subset of GSM8K (no network dependency)

In [ ]:
# Representative sample extracted from openai/gsm8k (train split).
GSM8K_SAMPLES = [
    {
        "question": (
            "Janet's ducks lay 16 eggs per day. She eats three for breakfast every "
            "morning and bakes muffins for her friends every day with four. She sells "
            "the remainder at the farmers' market daily for $2 per fresh duck egg. "
            "How much in dollars does she make every day at the farmers' market?"
        ),
        "answer": "18",
    },
    {
        "question": (
            "A robe takes 2 bolts of blue fiber and half that much white fiber. "
            "How many bolts in total does it take?"
        ),
        "answer": "3",
    },
    {
        "question": (
            "Josh decides to try flipping a house. He buys a house for $80,000 and "
            "then puts in $50,000 in repairs. This increased the value of the house "
            "by 150%. How much profit did he make?"
        ),
        "answer": "70000",
    },
]

## Callables required by tree_of_thoughts

In [ ]:
async def generate_solution_steps(
    problem: str,
    state: dict,
    path_so_far: list,
) -> list[str]:
    """Generate N candidate solution steps for the math problem.

    Args:
        problem: Problem statement (or accumulated partial step).
        state: Opaque state (contains the original statement in state["question"]).
        path_so_far: Previous thoughts in the current path.

    Returns:
        List of texts with different solution approaches.
    """
    settings = get_settings()
    llm = ProviderRegistry(settings=settings).get_llm()

    context = ""
    if path_so_far and len(path_so_far) > 1:
        # Include the previous path as refinement context
        prev = "\n".join(f"  Step {i}: {t.content[:200]}" for i, t in enumerate(path_so_far[1:], 1))
        context = f"\n\nPreviously explored steps:\n{prev}"

    system_prompt = (
        "You are an expert in mathematics. Given a problem, propose 3 "
        "different and concise approaches to solve it step by step. "
        "Return exactly 3 approaches separated by '|||'. "
        "Each approach must be self-contained and include the final numeric result."
    )
    user_prompt = f"Problem: {state.get('question', problem)}{context}"

    response = await llm.ainvoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ]
    )

    raw = str(response.content).strip()
    # Split the 3 candidates by the delimiter
    candidates = [c.strip() for c in raw.split("|||") if c.strip()]
    # Guarantee at least 1 candidate
    if not candidates:
        candidates = [raw]
    return candidates[:3]  # at most 3


async def evaluate_solution(thought_text: str, state: dict) -> float:
    """Score a solution thought in [0, 1].

    Criteria:
    - Contains a clearly identifiable final number → +0.4
    - That number matches the expected answer → +0.5
    - The solution has explicit reasoning (>50 chars) → +0.1

    Args:
        thought_text: Text of the thought to evaluate.
        state: Contains 'answer' with the expected solution.

    Returns:
        Score in [0.0, 1.0].
    """
    score = 0.0
    expected = state.get("answer", "").strip()

    # Extract numbers from the text
    numbers = re.findall(r"\b\d+(?:[.,]\d+)?\b", thought_text)
    if numbers:
        score += 0.4
        # Check whether the expected answer is among the numbers found
        normalized_expected = expected.replace(",", "").replace(".", "")
        for n in numbers:
            normalized_n = n.replace(",", "").replace(".", "")
            if normalized_n == normalized_expected:
                score += 0.5
                break

    # Reasoning quality: reasonable length
    if len(thought_text) > 50:
        score += 0.1

    return min(1.0, score)


## Main function

In [ ]:
async def solve_with_tot(sample: dict, strategy: str = "beam") -> ToTResult:
    """Solve a GSM8K problem using Tree of Thoughts.

    Args:
        sample: Dictionary with 'question' and 'answer'.
        strategy: Search strategy: 'beam', 'bfs' or 'dfs'.

    Returns:
        ToTResult with the best thought and the full path.
    """
    state = {
        "question": sample["question"],
        "answer": sample["answer"],
        "messages": [],
        "metadata": {},
    }

    return await tree_of_thoughts(
        problem=sample["question"],
        generate_fn=generate_solution_steps,
        evaluate_fn=evaluate_solution,
        state=state,
        breadth=3,  # 3 candidates per node
        depth=3,  # max depth of 3 levels
        beam_size=2,  # keep top-2 in beam search
        threshold=0.9,  # stop if any thought scores >= 0.9
        search_strategy=strategy,  # type: ignore[arg-type]
    )


async def main() -> None:
    print("=" * 70)
    print("  Tree of Thoughts — Dataset: GSM8K (Grade School Math)")
    print("=" * 70)

    for i, sample in enumerate(GSM8K_SAMPLES, 1):
        print(f"\n[Problem {i}]")
        print(f"  Question : {sample['question'][:80]}...")
        print(f"  Expected answer: {sample['answer']}")

        # Try with beam strategy (default)
        result = await solve_with_tot(sample, strategy="beam")

        print(f"\n  Best thought (score={result.best_thought.score:.2f}):")
        print(f"    {result.best_thought.content[:300]}")
        print(f"  Total thoughts generated: {result.total_thoughts_generated}")
        print(f"  Depth of best path: {len(result.best_path) - 1}")

        # Show the full path
        print("  Reasoning path:")
        for step in result.best_path:
            prefix = "  ROOT" if step.depth == 0 else f"  D{step.depth}"
            print(f"    {prefix} [score={step.score:.2f}] {step.content[:100]}...")

        print("-" * 70)

    # Strategy comparison on the first problem
    print("\n[Strategy comparison — Problem 1]")
    for strat in ["beam", "dfs", "bfs"]:
        r = await solve_with_tot(GSM8K_SAMPLES[0], strategy=strat)
        print(
            f"  {strat:5s}: score={r.best_thought.score:.2f}  "
            f"thoughts={r.total_thoughts_generated}"
        )


if __name__ == "__main__":
    asyncio.run(main())


---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()